



The formulas and terminology implemented in this code are teken from the article: Mohammed Jameel and Mohamed Abouhawwash: A new proximity metric based on optimality conditions for single and multi-objective optimization: Method and validation.

**Notebook content:**


**I. Tool functions:**

1.  **Define a function to evaluate constraints for a single solution x**

2.  **Dynamically define constraint functions g\_i(x)**
3.  **Define the main objective function F(x) used inside the PBI measure**
4.  **Define the PBI function**
    * Note that there are three possibilities for the PBI function: taking normal, smooth, or smooth\_pseudo\_huber.
5.  **Define phi(x) = PBI(...) for use in the KKTPM**


**II. Evolutionary algorihm setup:**

6.  **Setup the NSGA2 algorithm**
    *DistributionLogger class is used from the file DistributionLogger_class*

**III. Result plotting:**

7. **Evaluate KKTPM on the PF points \*\*if\*\* they are in decision space**
    * Note that we have a choice of the variables: `direction_vector` (w), `minimal_point` (r^star), and `sigma`.
8. **Heatmap function**
    * In the final version this function is not used, since it gives valuable information only in the case where objective space is two dimensional.
9. **Plot the population snapshots if the problem has at least 2 objectives**
10. **Plot the PF's KKTPM values** (if we have them)

*Outside the `compute_kktpm_for_problem` Function (Main Execution):*

**IV. Main execution:**

11. **"Main" part of code where we execute the compute\_kktpm\_for\_problem(problem) for every problem from the file optimization\_problems.py**

In [ ]:


# Pymoo imports
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from scipy.optimize import approx_fprime
from pymoo.problems import get_problem
from pymoo.core.problem import Problem
import traceback



# other imports
from pbi_proximity_measure import pbi_kktpm
from apbi_proximity_measure import apbi_kktpm
from optimization_problems import my_problems, pymoo_problems
from DistributionLogger_class import DistributionLogger
import numpy as np
import matplotlib.pyplot as plt



def compute_kktpm_for_problem(problem, total_evaluations=70, pop_size=50):
    """
    Compute and log (A)PBI-KKTPM metrics over multiple generations using NSGA2.

    Parameters
    ----------
    problem : Problem
        A pymoo Problem instance. Must define problem._evaluate(x, out) such that
        'out["F"]' has the objective values and optionally 'out["G"]', 'out["H"]'
        for constraints.
    total_evaluations : int, optional (default=70)
        The number of total generations (n_gen) for the minimization.
    pop_size : int, optional (default=50)
        The population size for NSGA2.

    Notes
    -----
    2. The code assumes that `problem.pareto_front()` is available and returns
       a 2D NumPy array of shape (n_points, n_obj). If `pareto_front()` is not
       implemented or returns None, it will cause errors.
    3. If the problem is single-objective, the second objective is forcibly set to 0
       for plotting.
    4. If the problem has more than 2 objectives, only the first 2 objectives are
       plotted.
    """

  
    # 1) Define a function to evaluate constraints for a single solution x

    def evaluate_constraints(x):
        """
        Evaluate constraints for a single solution x (which is 2D: shape (1, n_var)).

        Returns
        -------
        np.ndarray
            1D array of combined inequality constraints [g(x), h(x), -h(x)] if they exist.
            Empty array if no constraints exist.
        """
        out = {}
        problem._evaluate(x, out)

        # If out has "G", store it; else create empty array of shape (1, 0)
        g_arr = np.array(out["G"]) if "G" in out else np.empty((x.shape[0], 0))
        # If out has "H", store it; else create empty array
        h_arr = np.array(out["H"]) if "H" in out else np.empty((x.shape[0], 0))

        print("G constraints:", g_arr)
        print("H constraints:", h_arr)

        # Transform equality constraints H(x) = 0 into two inequalities: H(x) <= 0 and -H(x) <= 0
        if h_arr.size > 0:
            transformed_h = np.column_stack([h_arr, -h_arr])
        else:
            transformed_h = np.empty((x.shape[0], 0))

        # Combine G and transformed H
        if g_arr.size > 0 and transformed_h.size > 0:
            combined_constraints = np.column_stack([g_arr, transformed_h])
        elif g_arr.size > 0:
            combined_constraints = g_arr
        else:
            combined_constraints = transformed_h

        # If constraints exist, return the constraints of the first row; otherwise return empty
        if combined_constraints.size > 0:
            print("All constraints are:", combined_constraints[0])
            print("Output:", combined_constraints[0])
            return combined_constraints[0]
        else:
            print("All constraints are: None (empty)")
            print("Output: []")
            return np.empty((0,))


    # 2) Dynamically define constraint functions g_i(x)


    num_constraints = problem.n_constr

    if num_constraints == 0:
        num_constraints = 1

    def single_constraint_func(x, idx):
        """
        Evaluate the idx-th constraint for a single solution x.
        """
        # Make sure x is 2D for problem._evaluate
        X2D = x.reshape(1, -1)
        return evaluate_constraints(X2D)[idx]

    g_functions = []
    for i in range(num_constraints):
        # Use default argument trick: i=i prevents late binding in lambda
        g_functions.append(lambda zz, i=i: single_constraint_func(zz, i))


    # 3) Define the main objective function F(x) used inside the PBI measure

    def evaluate_objectives(x):
        """
        Evaluate the objectives for a single solution x (shape: (n_var,)).
        Returns a 1D array: out["F"][0].
        """
        # Make x 2D
        x_2d = np.array(x)[np.newaxis, :]
        out = {}
        problem._evaluate(x_2d, out)
        # out["F"] is expected to be shape (1, n_obj)
        return out["F"][0]


    # 4) Define the PBI function - There is a slight adaptation from the paper: the fact is that normally pbi function is not necessarily continuous. Therefore I checked two new versions in pbi_smooth and d2_smooth_pseudo_huber.

    def PBI(F_x, w, r_star, sigma):
        """
        Calculate the PBI value for a given solution.

        Parameters
        ----------
        F_x : np.ndarray
            Objective values of x. (1D)
        w : np.ndarray
            Weight vector (reference direction).
        r_star : np.ndarray
            Reference point with minimal values for each objective.
        sigma : float
            User-defined penalty factor.

        Returns
        -------
        float
            The PBI value for the solution x.
        """
        w = np.array(w)
        w = w / np.linalg.norm(w)  # Normalize the weight vector
        r_star = np.array(r_star)
        F_x = np.array(F_x)
        #Print out the values:

        print("F_x:", F_x)
        print("w:", w)
        print("r_star:", r_star)

        # d1: projection of (F(x) - r_star) onto w
        d1 = np.dot(F_x - r_star, w) / np.linalg.norm(w)
        # d2: perpendicular distance from F(x) to the line L

        beta = 10 
        delta = 1.0
        diff_sq = (F_x - (r_star + (d1 * w)))
        d2 = np.linalg.norm(diff_sq)
        #d2_smooth = (1 / beta) * np.log(np.sum(np.exp(beta * diff_sq))) - (1/beta) * np.log(len(F_x))
        d2_smooth_pseudo_huber = np.sum(delta**2 * (np.sqrt(1.0 + (diff_sq / delta)**2) - 1.0))


    

        return d1 + sigma * d2_smooth_pseudo_huber
    
    # Define function that returns d1 and another that returns d2

    def d1(x, w, r_star):
        F_x = evaluate_objectives(x)
        w = np.array(w)
        w = w / np.linalg.norm(w)
        r_star = np.array(r_star)
        F_x = np.array(F_x)
        # d1: projection of (F(x) - r_star) onto w
        d1 = np.dot(F_x - r_star, w) / np.linalg.norm(w)
        return d1
    
    def d2(x, w, r_star):
        F_x = evaluate_objectives(x)
        w = np.array(w)
        w = w / np.linalg.norm(w)
        r_star = np.array(r_star)
        F_x = np.array(F_x)
        # d2: perpendicular distance from F(x) to the line L
        d1 = np.dot(F_x - r_star, w) / np.linalg.norm(w)
        beta = 10  # Higher values make approximation closer to the original norm
        diff_sq = (F_x - (r_star + (d1 * w)))**2
        d2_smooth = (1 / beta) * np.log(np.sum(np.exp(beta * diff_sq)))
        return d2_smooth
    
    

    # 5) Define phi(x) = PBI(...) for use in the KKTPM

    def phi_func(x, direction_vector_1, ideal_point, sigma = 1):
        """
        An objective-like function that uses PBI to measure how close x is
        to a reference direction from 'ideal_point' in objective space.
        """
        f_vals = evaluate_objectives(x)
        return PBI(f_vals, w=direction_vector_1, r_star=ideal_point, sigma=sigma)






             




    # 6) Setup the NSGA2 algorithm and run the optimization



    num_parts = 6
    part_interval = total_evaluations // num_parts  # Evaluate every 'part_interval' generations

    algorithm = NSGA2(pop_size=pop_size)
    logger = DistributionLogger(part_interval, problem, problem, phi_func, g_functions)

    # Execute the minimization
    res = minimize(
        problem,
        algorithm,
        termination=('n_gen', total_evaluations),
        seed=1,
        callback=logger
    )

    # Retrieve the known Pareto front
    pf = problem.pareto_front()
    pareto_set = problem.pareto_set()
    if pareto_set is None or pf is None:
        print("Pareto front or set is None; skipping plotting.")
        return None




    # 7) Evaluate KKTPM on the PF points **if** they are in decision space



    def evaluate_kktpm_on_pf_decision_space(pf_decision, direction_vector_1, minimal_point, offset=1e-2):
        """
        Evaluate KKTPM for each row in `pf_decision`, assuming each row is x in R^n.
        """
        results = []
        exact_results = []
        grad = []
        for x_dec in pf_decision:
            val = apbi_kktpm(
                x_dec,
                lambda xx: phi_func(xx, direction_vector_1, minimal_point - offset),
                g_functions,
                implementation="complicate_linear_system"
            )
            exact_val = pbi_kktpm(
                x_dec,
                lambda xx: phi_func(xx, direction_vector_1, minimal_point - offset),
                g_functions
            )
            grad_d1 = approx_fprime(x_dec, lambda x: d1(x, direction_vector_1, minimal_point - offset), 1e-6)
                    
            grad_d2 = approx_fprime(x_dec, lambda x: d2(x, direction_vector_1, minimal_point - offset), 1e-6)
            if val is not None:
                results.append(val)
            else:
                results.append(10.0)
            if exact_val is not None:
                exact_results.append(exact_val)
            else:
                exact_results.append(10.0)
            grad.append([grad_d1[0], grad_d2[0]])
        return results, exact_results, grad

    # A quick check: is PF the same dimension as the decision space?
    # If so, we attempt to evaluate KKTPM at each PF point.
    pf_kkt_vals = None
    if pf.shape[1] == problem.n_obj:
        print(f"Detected PF is shape {pf.shape}, same as decision dimension => evaluating KKTPM on PF.")
        minimal_point = np.min(pf, axis=0) if pf.size else np.zeros(problem.n_obj)
        minimal_point = np.zeros(problem.n_obj)
        direction_vector_1 = np.ones(problem.n_obj)/problem.n_obj   # or any other reference direction
        pf_kkt_vals, pf_exact_kkt_vals, gradients = evaluate_kktpm_on_pf_decision_space(pareto_set, direction_vector_1, minimal_point = minimal_point)
        print("KKTPM at PF points:", pf_kkt_vals)
        print("Exact KKTPM at PF points:", pf_exact_kkt_vals)
    else:
        print("PF dimension != decision dimension => skipping KKTPM on PF points.")



    # 8) Heatmap function



    def evaluate_kktpm_on_grid(
        problem,
        direction_vector_1,
        minimal_point,
        g_functions,
        density=50,
        offset=1e-2
    ):
        """
        Creates a 2D grid in the domain [problem.xl, problem.xu],
        evaluates KKTPM at each grid point, and plots a heatmap.

        Parameters
        ----------
        problem : object
            An object that has:
              - problem.xl : np.array of shape (2,) (lower bounds)
              - problem.xu : np.array of shape (2,) (upper bounds)
        direction_vector_1 : np.ndarray
            Direction used inside phi_func (the PBI direction, for example).
        minimal_point : np.ndarray
            Minimal point used inside phi_func (the reference offset).
        g_functions : list of callables
            List of constraint functions g_j(x).
        density : int, optional
            Number of points in each dimension of the grid (default = 50).
        offset : float, optional
            Offset subtracted from minimal_point inside phi_func (default = 1e-2).

        Returns
        -------
        (X, Y, Z) : tuple of np.ndarray
            X, Y are the meshgrid arrays for the domain.
            Z contains the KKTPM measure evaluated at each grid point.
        """

        # 1) Create the grid in the range [xl, xu]
        x_vals = np.linspace(problem.xl[0], problem.xu[0], density)
        y_vals = np.linspace(problem.xl[1], problem.xu[1], density)
        X, Y = np.meshgrid(x_vals, y_vals)

        # 2) Prepare an array Z (same shape) to store KKTPM values
        Z = np.zeros_like(X, dtype=float)

        # 3) Define a small wrapper for phi_func that incorporates direction_vector_1 & minimal_point
        def phi_wrapper(xx):
            return phi_func(xx, direction_vector_1, minimal_point - offset)

        # 4) Evaluate KKTPM at each grid point
        for i in range(density):
            for j in range(density):
                x_dec = np.array([X[i, j], Y[i, j]])
                val = apbi_kktpm(
                    x_dec,
                    phi_wrapper,
                    g_functions,
                    implementation="complicate_linear_system"
                )
                # If val is None (not feasible or something else), assign a large fallback
                Z[i, j] = 10.0 if val is None else val

        # 5) Plot the heatmap using contourf or imshow
        plt.figure(figsize=(7, 5))
        contour_levels = 50  # You can change the number of contour levels if desired
        cs = plt.contourf(X, Y, Z, levels=contour_levels, cmap='viridis')
        plt.colorbar(cs, label="KKTPM value")
        plt.xlabel("x1")
        plt.ylabel("x2")
        plt.title("KKTPM Heatmap in Decision Space")
        plt.show()

        return X, Y, Z



    # 9) Plot the population snapshots if the problem has at least 2 objectives

    if problem.n_obj > 1 and pf.shape[1] > 1:
        plt.figure(figsize=(18, 12))

        # logger.data = [(iteration, [f1...], [f2...]), ...]
        for idx, (iteration, f1_vals, f2_vals) in enumerate(logger.data, start=1):
            plt.subplot(2, 3, idx)
            plt.scatter(f1_vals, f2_vals, label=f"Iteration {iteration}")
            plt.plot(pf[:, 0], pf[:, 1], label="Pareto Front", color="red", linewidth=2)
            plt.xlabel("$f_1(x)$")
            plt.ylabel("$f_2(x)$")
            plt.title(f"Population at Iteration {iteration}")
            plt.legend()

        plt.show()
    
    def to_float(sez):
        return [float(i) for i in sez]

    # Align KKTPM_measures and GD_measures by iteration numbers
    kkptm_dict = dict(logger.KKTPM_measures)  # Convert KKTPM_measures to a dictionary
    pbi_kkptm_dict = dict(logger.pbi_kktpm_measures)  # Convert KKTPM_measures to a dictionary
    gd_dict = dict(logger.GD_measures)       # Convert GD_measures to a dictionary

    # Ensure iterations are aligned
    common_iterations = sorted(set(kkptm_dict.keys()) & set(gd_dict.keys()))
    aligned_kktpm = [kkptm_dict[iteration] for iteration in common_iterations]
    aligned_pbi_kktpm = [pbi_kkptm_dict[iteration] for iteration in common_iterations]
    aligned_gd = [gd_dict[iteration] for iteration in common_iterations]
    
    aligned_kktpm_float = [to_float(i) for i in aligned_kktpm]
    aligned_pbi_kktpm_float = [to_float(i) for i in aligned_pbi_kktpm]



    iterations = list(range(len(gradients)))
    first_gradients = [value[0] for value in gradients]
    second_gradients = [value[1] for value in gradients]

    # Plot the gradients
    plt.figure(figsize=(8, 6))

    plt.plot(iterations, first_gradients, 'ro-', label='First Derivative')
    plt.plot(iterations, second_gradients, 'bo-', label='Second Derivative')

    # Adding labels and legend
    plt.xlabel('Iteration')
    plt.ylabel('Derivative Value')
    plt.title('First and Second Derivatives Over Iterations')
    plt.legend()
    plt.grid(True)

    # Show the plot
    plt.show()

    
    # Plot 1: KKTPM Measures (Box Plots + Median Line)
    plt.figure(figsize=(12, 8))

    # Create box plots for KKTPM
    box = plt.boxplot(aligned_kktpm_float, labels=common_iterations, patch_artist=True)

    # Calculate and plot medians for KKTPM
    medians = [np.median(group) for group in aligned_kktpm_float]
    plt.plot(range(1, len(medians) + 1), medians, color='red', marker='o', linestyle='-', label='KKTPM Median')

    # Add labels and title
    plt.yscale("log")
    plt.xlabel('Iterations')
    plt.ylabel('KKTPM Values')
    plt.title('Box plot and Median Line for KKTPM measures at each iteration')
    plt.legend()

    # Show the plot
    plt.show()

    # Create box plots for exact KKTPM
    box = plt.boxplot(aligned_pbi_kktpm_float, labels=common_iterations, patch_artist=True)
    # Calculate and plot medians for exact KKTPM
    medians = [np.median(group) for group in aligned_pbi_kktpm_float]
    plt.plot(range(1, len(medians) + 1), medians, color='red', marker='o', linestyle='-', label='KKTPM Median')

    # Add labels and title
    plt.yscale("log")
    plt.xlabel('Iterations')
    plt.ylabel('Exact KKTPM Values')
    plt.title('Box plot and Median Line for Exact KKTPM measures at each iteration')
    plt.legend()

    # Show the exact plot
    plt.show()

    # Plot 2: Generational Distance (GD)
    plt.figure(figsize=(12, 6))

    # Plot GD values
    plt.plot(
        common_iterations, aligned_gd,
        color='blue', marker='s', linestyle='--', label='Generational Distance (GD)'
    )

    # Add labels and title
    plt.yscale("log")
    plt.xlabel('Iterations')
    plt.ylabel('GD Values')
    plt.title('Generational Distance (GD) Over Iterations')
    plt.legend()

    # Show the second plot
    plt.show()


    # 10) **Plot the PF's KKTPM values** (if we have them)
    # add the exact kktpm values with smaller size of points


    print("Evaluate KKTPM on PF points \n\n\n\n\n\n\n\n\n\n\n")
    pf_kkt_vals, pf_exact_kkt_vals, grads = evaluate_kktpm_on_pf_decision_space(pareto_set, direction_vector_1, minimal_point = minimal_point)
    print("KKTPM at PF points:", pf_kkt_vals)

    if pf_kkt_vals is not None:
        plt.figure(figsize=(7, 5))
        plt.plot(pf_kkt_vals, 'o-', color='green', markersize=3)
        plt.yscale('log')
        plt.xlabel("Index of PF Point")
        plt.ylabel("KKTPM (log-scale)")
        plt.title("KKTPM Evaluated on Provided Pareto-Set Points")
        plt.show()

    if pf_exact_kkt_vals is not None:
        plt.figure(figsize=(7, 5))
        plt.plot(pf_exact_kkt_vals, 'o-', color='blue', markersize=3)
        plt.yscale('log')
        plt.xlabel("Index of exact PF Point")
        plt.ylabel("exact KKTPM (log-scale)")
        plt.title("exact KKTPM Evaluated on Provided Pareto-Set Points")
        plt.show()





# 11) "Main" part of code where we execute the compute_kktpm_for_problem(problem) for every problem from the file optimization_problems.py



for problem_name in pymoo_problems:
    print(f"\n\n\n\n\n\n\n\n\n________________________________________________________Problem: {problem_name}") 
    try:
        # Try to load the problem
        problem = get_problem(problem_name)
        
        # Print problem properties
        print(f"Number of variables: {problem.n_var}")
        print(f"Number of objectives: {problem.n_obj}")
        print(f"Number of constraints: {problem.n_constr}")

        # Run your function
        compute_kktpm_for_problem(problem, 250, pop_size=50)

    except Exception as e:
        # Handle the exception and continue
        print(f"An error occurred while loading problem '{problem_name}': {e}")
        traceback.print_exc()  # Print the full stack trace
    
    




for problem_name  in my_problems:
    print(f"\n\n\n\n\n\n\n\n\n________________________________________________________Problem: {problem_name}") 
    #try:
        # Try to load the problem
    problem = problem_name
    # Print problem properties
    print(f"Number of variables: {problem.n_var}")
    print(f"Number of objectives: {problem.n_obj}")
    print(f"Number of constraints: {problem.n_constr}")
    compute_kktpm_for_problem(problem, 40, pop_size=50)
        
    #except Exception as e:
    #    # Handle the exception and continue
    #    print(f"An error occurred while loading problem '{problem_name}': {e}")
    #    traceback.print_exc()  # Print the full stack trace



    